# 2차시: Multi-Agent (AutoGen) 실습 (Part 1)

AutoGen 기반 멀티 에이전트 시스템 구축 실습 노트북입니다.

## 목차
1. AutoGen 프레임워크 개요
2. 설치 및 기본 설정
3. Agent 구성요소
4. RoundRobinGroupChat (순환 대화)
5. SelectorGroupChat (동적 선택)
6. Swarm (자율 핸드오프)

## 환경 설정

AutoGen 패키지 설치 및 API 키 설정

In [ ]:
# AutoGen 설치 (최초 1회)
# !pip install -U "autogen-agentchat" "autogen-ext[openai]"

In [ ]:
import os

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o")

---
## 3. Agent 구성요소

### 3-1. AssistantAgent 생성 및 실행

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console

agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    system_message="You are a helpful AI assistant.",
)

In [ ]:
# Method 1: Get result directly
result = await agent.run(task="Tell me 3 advantages of Python")
print(result.messages[-1].content)

In [ ]:
# Method 2: Streaming output
await Console(agent.run_stream(task="Tell me 3 advantages of Python"))

### 3-2. UserProxyAgent

In [ ]:
from autogen_agentchat.agents import UserProxyAgent

# Console input proxy
user_proxy = UserProxyAgent("user_proxy", input_func=input)

### 3-3. Message Types

In [ ]:
from autogen_agentchat.messages import TextMessage, HandoffMessage

msg = TextMessage(content="Hello!", source="User")
print(msg)

handoff = HandoffMessage(
    source="travel_agent",
    target="refund_agent",
    content="Please process the refund.",
)
print(handoff)

---
## 4. RoundRobinGroupChat (순환 대화)

### 4-1. Writer + Critic (Reflection Pattern)

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console

writer = AssistantAgent(
    "writer",
    model_client=model_client,
    system_message="You are a creative writer. Write based on the given topic.",
)
critic = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="""You are a writing critic. Review the writing and provide specific feedback.
    If satisfied, respond with 'APPROVE'.""",
)

termination = TextMentionTermination("APPROVE")

team = RoundRobinGroupChat([writer, critic], termination_condition=termination)
await Console(team.run_stream(task="Write a short poem about autumn"))

---
## 5. SelectorGroupChat (동적 선택)

### 5-1. Multi-Expert Collaboration

In [ ]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

planner = AssistantAgent(
    "PlanningAgent",
    description="Agent that plans tasks. Should participate first for new tasks.",
    model_client=model_client,
    system_message="Break down complex tasks into sub-tasks.",
)
researcher = AssistantAgent(
    "ResearchAgent",
    description="Agent that researches and gathers information.",
    model_client=model_client,
    system_message="Research and return relevant information on the given topic.",
)
analyst = AssistantAgent(
    "AnalystAgent",
    description="Agent that analyzes data and performs calculations.",
    model_client=model_client,
    system_message="Analyze data and compute results. When the final report is complete, respond with 'TERMINATE'.",
)

In [ ]:
termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(25)

team = SelectorGroupChat(
    [planner, researcher, analyst],
    model_client=model_client,
    termination_condition=termination,
    allow_repeated_speaker=True,
)

await Console(team.run_stream(
    task="Analyze the recent trends in generative AI technology"
))

### 5-2. Custom Selector Function

In [ ]:
from typing import Sequence
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage

def custom_selector(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    """Custom routing: always go through Planner first"""
    if messages[-1].source != "PlanningAgent":
        return "PlanningAgent"
    return None  # Fallback to LLM-based selection

team = SelectorGroupChat(
    [planner, researcher, analyst],
    model_client=model_client,
    termination_condition=termination,
    selector_func=custom_selector,
)

await Console(team.run_stream(
    task="Compare the pros and cons of Python vs JavaScript"
))

---
## 6. Swarm (자율 핸드오프)

### 6-1. Customer Service Swarm

In [ ]:
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination, TextMentionTermination

def refund_flight(flight_id: str) -> str:
    """Process a flight refund."""
    return f"Flight {flight_id} has been refunded successfully."

travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""You are a travel consultation agent. Delegate refund requests to flights_refunder.
    If you need user information, handoff to user. When done, respond with TERMINATE.""",
)

flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent", "user"],
    tools=[refund_flight],
    system_message="""You are a refund specialist agent. Use the refund_flight tool to process refunds.
    After completion, handoff back to travel_agent.""",
)

In [ ]:
termination = (
    HandoffTermination(target="user")
    | TextMentionTermination("TERMINATE")
)

team = Swarm(
    [travel_agent, flights_refunder],
    termination_condition=termination,
)

result = await Console(
    team.run_stream(task="I would like to request a refund for flight FL-12345.")
)

### 6-2. Swarm with User Handoff Loop

In [ ]:
from autogen_agentchat.messages import HandoffMessage

# Reset team state for a fresh conversation
await team.reset()

result = await Console(
    team.run_stream(task="I need help with my flight booking.")
)
last = result.messages[-1]

# Handle user handoff loop
while isinstance(last, HandoffMessage) and last.target == "user":
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    result = await Console(team.run_stream(
        task=HandoffMessage(
            source="user", target=last.source, content=user_input
        )
    ))
    last = result.messages[-1]